# **I. DATASET: BERLIN**
# BỘ DỮ LIỆU CHUẨN: BERLIN52 (TSPLIB)

> **Giới thiệu:** `berlin52` là bộ dữ liệu thuộc thư viện **TSPLIB** (Đại học Heidelberg), bao gồm tọa độ của **52 địa điểm tại thành phố Berlin, Đức**. Bộ dữ liệu này thường được sử dụng làm benchmark để đánh giá, so sánh độ chính xác và hiệu năng của các thuật toán tối ưu hóa (Greedy, GA, ACO, Simulated Annealing,...).

---

### Thông Số Kỹ Thuật Của Dataset

Để đảm bảo tính chính xác khi lập trình, dưới đây là các thông tin cấu hình phần cứng/phần mềm của bộ dữ liệu:

| Thuộc tính | Giá trị chi tiết | Ý nghĩa |
| :--- | :--- | :--- |
| **Name** | `berlin52` | Tên định danh trong thư viện TSPLIB |
| **Type** | `TSP` | Bài toán Người giao hàng đối xứng |
| **Dimension** | `52` | Tổng số lượng thành phố (nút đồ thị) |
| **Edge Weight Type** | `EUC_2D` | Khoảng cách Euclid phẳng (Hệ tọa độ 2D) |
| **Best Known Optimal** | $\mathbf{7542}$ | **Độ dài quãng đường ngắn nhất tối ưu** |

---


## **1. Repeated Nearest Neighbor + Relocate**

In [1]:
import math
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

###  Cấu Trúc Dữ Liệu & Khởi Tạo Ma Trận Khoảng Cách

Lớp `Instance` chịu trách nhiệm lưu trữ tọa độ các thành phố và tự động tính toán **Ma trận khoảng cách đối xứng (Distance Matrix)** theo chuẩn **TSPLIB**.

* **`self.nodes`**: Danh sách lưu tọa độ các đỉnh dưới dạng `[(x1, y1), (x2, y2), ...]`.
* **`euclid_distance(u, v)`**: Tính khoảng cách hình học giữa hai thành phố $u$ và $v$, sử dụng hàm `round()` để làm tròn thành số nguyên theo đúng quy ước của bộ dữ liệu `berlin52`.
* **`build_distance_matrix()`**: Khởi tạo và điền dữ liệu vào ma trận vuông hai chiều `self.dist` kích thước $N \times N$, giúp tối ưu hóa thời gian tra cứu khoảng cách từ $O(1)$ thay vì phải tính lại nhiều lần khi chạy thuật toán.

$$\text{dist}[i][j] = \text{round}\left(\sqrt{(x_i - x_j)^2 + (y_i - y_j)^2}\right)$$

In [2]:
class Instance:
    def __init__(self):
        self.nodes = []
        self.dist  = []

    def euclid_distance(self, u, v):
        xu, yu = self.nodes[u]
        xv, yv = self.nodes[v]
        xd = xu - xv
        yd = yu - yv
        return round(math.sqrt(xd * xd + yd * yd))  # Lam tron nguyen theo quy uoc TSPLIB

    def build_distance_matrix(self):
        n = len(self.nodes)
        self.dist = [[0] * n for _ in range(n)]
        for i in range(n):
            for j in range(i + 1, n):
                d = self.euclid_distance(i, j)
                self.dist[i][j] = d
                self.dist[j][i] = d

In [3]:
# Hàm đọc file
def read_tsp_file(filepath):
    instance = Instance()
    in_coord_section = False

    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line == "NODE_COORD_SECTION":
                in_coord_section = True
                continue
            if line in ("EOF", ""):
                continue
            if line[0].isalpha() and ":" in line:
                continue
            if in_coord_section:
                parts = line.split()
                x, y = float(parts[1]), float(parts[2])
                instance.nodes.append((x, y))
    instance.build_distance_matrix()
    return instance

In [4]:
def tour_length(tour, dist):
    n = len(tour)
    total = 0
    for i in range(n):
        total += dist[tour[i]][tour[(i + 1) % n]]
    return total


###  **Kiểm Tra Tính Hợp Lệ Của Lộ Trình (Tour Validation)**

Hàm bổ trợ giúp đảm bảo nghiệm (đường đi) tìm được từ các thuật toán Heuristic tuân thủ nghiêm ngặt các quy tắc toán học của bài toán TSP trước khi tiến hành tính toán chi phí.

* **`is_valid_tour(tour, n)`**: Kiểm tra hai điều kiện cốt lõi của một chu trình TSP:
  1. Số lượng thành phố đi qua phải chính xác bằng $N$ (`len(tour) == n`).
  2. Mọi thành phố trong danh sách từ `0` đến `n-1` phải xuất hiện **đúng một lần** (không thiếu, không trùng lặp).
* **`validate_solution(tour, instance, label)`**: Wrapper bao bọc hàm kiểm tra. Nếu phát hiện lộ trình lỗi, hàm sẽ lập tức đưa ra cảnh báo trực quan `[!!! ERROR !!!]` và ném ra ngoại lệ (`ValueError`) để ngăn chặn việc sử dụng dữ liệu sai trong các bước tiếp theo.

In [5]:
def is_valid_tour(tour, n):
    if len(tour) != n:
        return False, f"Tour have {len(tour)} nodes, need {n}"
    if sorted(tour) != list(range(n)):
        missing = set(range(n)) - set(tour)
        return False, f"Tour is missing nodes: {missing}"
    return True, "Valid tour"

def validate_solution(tour, instance, label=""):
    n = len(instance.nodes)
    valid, msg = is_valid_tour(tour, n)
    if not valid:
        print(f"  [!!! ERROR !!!] {label}: {msg}")
        raise ValueError(f"Lộ trình {label} không hợp lệ!")
    return valid

##  **Khởi tạo các hàm cho Greedy & Local Search**

In [6]:
def nearest_neighbor(dist, start, n):
    visited = [False] * n
    tour = [start]
    visited[start] = True
    current = start

    for _ in range(n - 1):
        best_dist = math.inf
        best_next = -1
        row = dist[current]
        for j in range(n):
            if not visited[j] and row[j] < best_dist:
                best_dist = row[j]
                best_next = j
        tour.append(best_next)
        visited[best_next] = True
        current = best_next

    return tour, tour_length(tour, dist)


def repeated_nearest_neighbor(instance):
    n = len(instance.nodes)
    dist = instance.dist
    best_tour = None
    best_cost = math.inf
    best_start = 0

    for start in range(n):
        tour, cost = nearest_neighbor(dist, start, n)
        if cost < best_cost:
            best_cost = cost
            best_tour = tour[:]
            best_start = start

    return best_tour, best_cost, best_start

In [7]:
def delta_relocate(tour, dist, i, n):
    prev_i = (i - 1) % n
    next_i = (i + 1) % n
    city = tour[i]
    prev_city   = tour[prev_i]
    next_city   = tour[next_i]
    remove_gain = -dist[prev_city][city] - dist[city][next_city] + dist[prev_city][next_city] # Khi xoa city, can loai bo 2 canh (p-city) va (city-nx), nhung them canh (p-nx) de tour van lien tuc.
    return remove_gain, city, prev_city, next_city

In [8]:
def relocate_first_improvement(instance, initial_tour):
    dist       = instance.dist
    n          = len(initial_tour)
    tour       = initial_tour[:]
    cost       = tour_length(tour, dist)
    iterations = 0
    improved   = True

    while improved:
        improved = False
        for i in range(n):
            remove_gain, city, _, _ = delta_relocate(tour, dist, i, n)
            tour_tmp = tour[:i] + tour[i+1:]
            m = n - 1

            for j in range(m):
                a = tour_tmp[j]
                b = tour_tmp[(j + 1) % m]
                insert_gain = -dist[a][b] + dist[a][city] + dist[city][b]
                delta = remove_gain + insert_gain

                if delta < 0:
                    tour_tmp.insert(j + 1, city)
                    tour  = tour_tmp
                    n     = len(tour)
                    cost += delta
                    iterations += 1
                    improved = True
                    break

            if improved:
                break

    return tour, cost, iterations

In [9]:
def relocate_best_improvement(instance, initial_tour):
    dist       = instance.dist
    n          = len(initial_tour)
    tour       = initial_tour[:]
    cost       = tour_length(tour, dist)
    iterations = 0

    while True:
        best_delta = 0
        best_i     = -1
        best_j     = -1

        for i in range(n):
            remove_gain, city, _, _ = delta_relocate(tour, dist, i, n)
            tour_tmp = tour[:i] + tour[i+1:]
            m = n - 1

            for j in range(m):
                a = tour_tmp[j]
                b = tour_tmp[(j + 1) % m]
                insert_gain = -dist[a][b] + dist[a][city] + dist[city][b]
                delta = remove_gain + insert_gain

                if delta < best_delta:
                    best_delta = delta
                    best_i = i
                    best_j = j

        if best_i == -1:
            break

        city = tour.pop(best_i)
        tour.insert(best_j + 1, city)
        cost += best_delta
        iterations += 1

    return tour, cost, iterations

In [10]:
def _two_opt_delta(tour, dist, i, j, n):
    a  = tour[(i - 1) % n]
    b  = tour[i]
    c  = tour[j]
    d  = tour[(j + 1) % n]
    delta = (dist[a][c] + dist[b][d]) - (dist[a][b] + dist[c][d])
    return delta

In [11]:
def two_opt_first_improvement(instance, initial_tour):
    dist       = instance.dist
    n          = len(initial_tour)
    tour       = initial_tour[:]
    cost       = tour_length(tour, dist)
    iterations = 0
    improved   = True

    while improved:
        improved = False
        for i in range(1, n - 1):
            for j in range(i + 1, n):
                delta = _two_opt_delta(tour, dist, i, j, n)
                if delta < 0:
                    tour[i : j + 1] = tour[i : j + 1][::-1]
                    cost    += delta
                    iterations += 1
                    improved = True
                    break
            if improved:
                break

    return tour, cost, iterations

In [12]:
def two_opt_best_improvement(instance, initial_tour):
    dist       = instance.dist
    n          = len(initial_tour)
    tour       = initial_tour[:]
    cost       = tour_length(tour, dist)
    iterations = 0

    while True:
        best_delta = 0
        best_i     = -1
        best_j     = -1

        for i in range(1, n - 1):
            for j in range(i + 1, n):
                delta = _two_opt_delta(tour, dist, i, j, n)
                if delta < best_delta:
                    best_delta = delta
                    best_i = i
                    best_j = j

        if best_i == -1:
            break

        # Dao nguoc doan [best_i .. best_j]
        tour[best_i : best_j + 1] = tour[best_i : best_j + 1][::-1]
        cost += best_delta
        iterations += 1

    return tour, cost, iterations




**Tạo các hàm Visualization**

In [13]:
def plot_tour(instance, tour, title, cost, ax, color='steelblue'):
    coords = instance.nodes
    n = len(tour)
    for k in range(n):
        u = tour[k]
        v = tour[(k + 1) % n]
        x1, y1 = coords[u]
        x2, y2 = coords[v]
        ax.plot([x1, x2], [y1, y2], color=color, lw=1.2, alpha=0.7)
    for idx, city in enumerate(tour):
        x, y = coords[city]
        c  = 'tomato' if idx == 0 else 'white'
        sz = 120 if idx == 0 else 60
        ax.scatter(x, y, c=c, s=sz, zorder=5, edgecolors='gray', linewidths=0.5)
        ax.text(x, y, str(city + 1), fontsize=6, ha='center', va='center',
                color='black', weight='bold')
    ax.set_title(f"{title}\nTotal Distance: {cost:,}", fontsize=12, pad=6)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    ax.tick_params(labelsize=10)


def plot_comparison(instance, left, right, save_path,
                    suptitle=None, ref_cost=None):
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    if suptitle is None:
        suptitle = f"TSP — berlin52:  {left['label']}  vs  {right['label']}"
    fig.suptitle(suptitle, fontsize=16, fontweight='bold', y=1.01)

    def _make_title(side, is_right=False):
        title = f"[{side['label']}]"
        # title += f"Total Distance: {side['cost']:,}\n"
        if side.get('note'):
            title += f"{" " + "-" + " "+ side['note']}\n"
        if is_right and ref_cost is not None:
            imp = (ref_cost - side['cost']) / ref_cost * 100
            if imp >= 0:
                title += f"Improvement: -{imp:.2f}% (better)"
            else:
                title += f"Degradation: +{abs(imp):.2f}% (worse)"
        return title
    plot_tour(instance, left['tour'],   #rnn_tour
              _make_title(left), left['cost'],
              axes[0], color=left['color'])

    plot_tour(instance, right['tour'],  #rls_tour
              _make_title(right, is_right=True), right['cost'],
              axes[1], color=right['color'])


    legend_patch = mpatches.Patch(color='tomato', label='Start city (tour[0])')
    fig.legend(handles=[legend_patch], fontsize=8,
               loc='lower center', ncol=1, framealpha=0.7)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"  [Plot] Saved: {save_path}")
    plt.close(fig)


def plot_all_comparisons(instance, rnn_tour, rnn_cost, rnn_start,
                         results, out_dir=r"D:\Ngọc\FDA-NEU\Tài liệu kì II\Discrete Mathematics\Project\outputs"):
    idx = {r[0]: r for r in results}
    _, rel_bi_tour, rel_bi_cost, _ = idx["Relocate-BI"]
    _, rel_fi_tour, rel_fi_cost, _ = idx["Relocate-FI"]
    _, opt_bi_tour, opt_bi_cost, _ = idx["2opt-BI"]
    _, opt_fi_tour, opt_fi_cost, _ = idx["2opt-FI"]

    C_RNN     = "#EE3A1E"
    C_REL_BI  = '#1565C0'
    C_REL_FI  = '#FF8F00'
    C_2OPT_BI = '#2E7D32'
    C_2OPT_FI = '#6A1B9A'

    rnn_side = dict(tour=rnn_tour, cost=rnn_cost,
                    label=f"RNN (start city {rnn_start+1})",
                    color=C_RNN, note=None)

    pairs = [
        (
            rnn_side,
            dict(tour=rel_bi_tour, cost=rel_bi_cost,
                 label="RNN + Relocate-BI", color=C_REL_BI, note=" 12 iters"),
            "cmp_A__RNN_vs_Relocate_BI.png",
            "TSP - berlin52: Repeated Nearest Neighbor (RNN)  vs  RNN + Relocate-BI"
        ),
        (
            rnn_side,
            dict(tour=opt_fi_tour, cost=opt_fi_cost,
                 label="RNN + 2opt-FI", color=C_2OPT_FI, note=" 12 iters"),
            "cmp_B__RNN_vs_2opt_FI.png",
            "TSP - berlin52: Repeated Nearest Neighbor (RNN)  vs  RNN + 2opt-FI"
        ),
        (
            dict(tour=rel_bi_tour, cost=rel_bi_cost,
                 label="Relocate-BI", color=C_REL_BI, note=" 12 iters"),
            dict(tour=rel_fi_tour, cost=rel_fi_cost,
                 label="Relocate-FI", color=C_REL_FI, note=" 13 iters"),
            "cmp_C__Relocate_BI_vs_FI.png",
            "TSP - berlin52: RNN + Relocate-BI  vs  RNN + Relocate-FI"
        ),
        (
            dict(tour=opt_bi_tour, cost=opt_bi_cost,
                 label="2opt-BI", color=C_2OPT_BI, note=" 5 iters"),
            dict(tour=opt_fi_tour, cost=opt_fi_cost,
                 label="2opt-FI", color=C_2OPT_FI, note=" 12 iters"),
            "cmp_D__2opt_BI_vs_FI.png",
            "TSP - berlin52: RNN + 2opt-BI  vs RNN + 2opt-FI"
        ),
        (
            dict(tour=rel_bi_tour, cost=rel_bi_cost,
                 label="Relocate-BI", color=C_REL_BI, note=" 12 iters"),
            dict(tour=opt_bi_tour, cost=opt_bi_cost,
                 label="2opt-BI", color=C_2OPT_BI, note=" 5 iters"),
            "cmp_E__Relocate_vs_2opt__BI.png",
            "TSP - berlin52: RNN + Relocate-BI  vs  RNN + 2opt-BI"
        ),
        (
            dict(tour=rel_fi_tour, cost=rel_fi_cost,
                 label="Relocate-FI", color=C_REL_FI, note=" 13 iters"),
            dict(tour=opt_fi_tour, cost=opt_fi_cost,
                 label="2opt-FI", color=C_2OPT_FI, note=" 12 iters"),
            "cmp_F__Relocate_vs_2opt__FI.png",
            "TSP - berlin52: RNN + Relocate-FI  vs  RNN + 2opt-FI"
        ),
    ]

    saved = []
    for left_d, right_d, fname, title in pairs:
        path = f"{out_dir}/{fname}"
        plot_comparison(instance, left_d, right_d, path,
                        suptitle=title, ref_cost=left_d['cost'])
        saved.append(path)

    return saved

##  **Khởi Chạy Hệ Thống Giải TSP & Đánh Giá Hiệu Năng (Execution Pipeline)**

Hàm `main()` đóng vai trò là **luồng điều hướng trung tâm (Pipeline)** để thực thi thử nghiệm, tích hợp toàn bộ các module đã định nghĩa trước đó nhằm giải quyết bài toán `berlin52.tsp`.

###  Quy Trình Xử Lý Gồm 4 Bước:

1. **Đọc Dữ Liệu & Tạo Nghiệm Nền (Baseline):** Tải file cấu hình tọa độ, đồng thời chạy ngầm thuật toán **Repeated Nearest Neighbor (RNN)** để nhanh chóng tạo ra một lộ trình xuất phát ban đầu.
2. **Tối Ưu Hóa Đồng Loạt (Local Search Benchmark):** Đưa nghiệm RNN qua 4 biến thể tìm kiếm cục bộ khác nhau bao gồm **Relocate** và **2-opt** (theo cả hai chiến lược cải tiến tốt nhất - *Best Improvement* và cải tiến đầu tiên - *First Improvement*). mọi kết quả đều được kiểm định tính hợp lệ (`assert`) một cách tự động.
3. **Đánh Giá & Tính Toán Sai Số (Gap %):** Xuất bảng thống kê trực quan so sánh các thuật toán dựa trên ba tiêu chí: **Quãng đường**, **Thời gian thực thi (ms)**, và **Khoảng cách sai số (Gap %)** so với nghiệm tối ưu toàn cục ($7542$).
4. **Đóng Gói Dữ Liệu (Data Export):** Xác định cấu hình tốt nhất và đóng gói toàn bộ kết quả vào một biến toàn cục (`viz_data`), sẵn sàng chuyển giao dữ liệu cho bước vẽ biểu đồ trực quan hóa (Visualization).

In [14]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE  = r"Dataset/berlin52.tsp"
    OPTIMAL   = 7542

    # [1] Đọc dữ liệu
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # Khởi tạo tour: Repeated Nearest Neighbor (Chạy ngầm)
    t0_rnn = time.perf_counter()
    rnn_tour, rnn_cost, best_start = repeated_nearest_neighbor(instance)
    t_rnn = (time.perf_counter() - t0_rnn) * 1000

    # Validation
    assert is_valid_tour(rnn_tour, n)[0], "Lỗi: Tour RNN không hợp lệ!"

    # Chạy Local Search variants (Chạy ngầm)
    ls_configs = [
        ("Relocate-BI", relocate_best_improvement),
        ("Relocate-FI", relocate_first_improvement),
        ("2opt-BI",     two_opt_best_improvement),
        ("2opt-FI",     two_opt_first_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, rnn_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        # Silent Validation
        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [2] Bảng so sánh tổng hợp (Summary Comparison)
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<35} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<35} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    # In kết quả gốc
    _print_row("Repeated Nearest Neighbor (RNN)", rnn_cost, t_rnn)

    # Sắp xếp kết quả LS theo quãng đường tăng dần để dễ so sánh
    results.sort(key=lambda x: x[2])
    for label, _, cost, t_ms in results:
        _print_row(f"RNN + {label}", cost, t_ms)

    # Lưu dữ liệu vào biến global để cell Visualization sử dụng
    global viz_data
    viz_data = {
        'instance': instance,
        'rnn_tour': rnn_tour,
        'rnn_cost': rnn_cost,
        'best_start': best_start,
        'results': results
    }

    best_label, _, best_cost, _ = results[0]
    print(f"\n[3] Best Solution: RNN + {best_label} ({best_cost:,})")
    print("\n" + "=" * 70)
    print("PROCESS COMPLETE (All solutions validated silently)")
    print("=" * 70)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/berlin52.tsp...
    Cities (n): 52

[2] Summary Comparison (Optimal: 7,542):
  Method                                  Distance    Time (ms)      Gap %
  ------------------------------------------------------------------------
  Repeated Nearest Neighbor (RNN)            8,181        41.91      8.47%
  RNN + 2opt-FI                              7,669        23.06      1.68%
  RNN + 2opt-BI                              7,711         9.41      2.24%
  RNN + Relocate-BI                          7,895        83.33      4.68%
  RNN + Relocate-FI                          7,907        26.11      4.84%

[3] Best Solution: RNN + 2opt-FI (7,669)

PROCESS COMPLETE (All solutions validated silently)


In [15]:
# #Visualize
# # <h1 style="font-size:30px; font-weight:bold;">2. VISUALIZATION & EXPORT</h1>

# def run_visualization():
#     if 'viz_data' not in globals():
#         print("❌ Lỗi: Bạn cần chạy cell thuật toán trước khi thực hiện visualize!")
#         return

#     print("\n[V] Starting visualization process...")
#     OUT_DIR = r"."

#     # Lấy dữ liệu từ biến global
#     data = viz_data

#     try:
#         saved = plot_all_comparisons(
#             data['instance'],
#             data['rnn_tour'],
#             data['rnn_cost'],
#             data['best_start'],
#             data['results'],
#             out_dir=OUT_DIR
#         )
#         print(f"✅ Thành công: Đã lưu {len(saved)} biểu đồ vào thư mục '{OUT_DIR}'")

#         # Hiển thị lộ trình tốt nhất (nếu hàm plot có trả về đường dẫn hoặc show trực tiếp)
#         # plt.show() # Nếu hàm của bạn sử dụng matplotlib

#     except NameError:
#         print("❌ Lỗi: Hàm 'plot_all_comparisons' chưa được định nghĩa.")
#     except Exception as e:
#         print(f"❌ Có lỗi xảy ra: {e}")

# run_visualization()

# **2. Mở Rộng**
##  Phần Mở Rộng: Đánh Giá & So Sánh các Thuật Toán

Ở phần trước, chúng ta mới chỉ thử nghiệm với RNN và Relocate, phần này sẽ mở rộng phạm vi thử nghiệm bằng cách đưa nhiều thuật toán khác nhau lên bàn cân. Mục tiêu cốt lõi là so sánh đối chứng hiệu quả của từng phương pháp, từ đó tìm ra combo tối ưu nhất để giải bài toán TSP.

###  1. Các thuật toán Greedy
* **Nearest Insertion (NI) vs. Cheapest Insertion (CI):** Đánh giá xem việc ưu tiên chèn thành phố gần nhất hay việc tối thiểu hóa chi phí phát sinh ngay từ đầu sẽ tạo ra nghiệm nền (Baseline) chất lượng hơn.
* **Farthest Insertion (FI):** Kiểm chứng giả thuyết: Liệu việc xây dựng "khung xương" lộ trình từ các thành phố xa nhất có giúp thuật toán tránh bẫy tối ưu cục bộ tốt hơn các phương pháp khác hay không.

###  2. Các thuật toán Local Search
* **Node-based (Swap / Exchange) vs. Edge-based (2-opt):** So sánh giữa việc hoán đổi vị trí hai thành phố (thay đổi đỉnh) với việc đảo ngược một đoạn lộ trình (thay đổi cạnh).
* **First Improvement (FI) vs. Best Improvement (BI):** Đánh giá sự đánh đổi (Trade-off) giữa **tốc độ hội tụ** (FI - chọn ngay nghiệm tốt hơn đầu tiên) và **chất lượng nghiệm** (BI - duyệt toàn bộ để chọn nghiệm tốt nhất).

>  **Mục tiêu phân tích:** Xác định xem "cặp bài trùng" nào `[Greedy] + [Local Search]` sẽ mang lại sự cân bằng tốt nhất giữa **Thời gian thực thi (ms)** và **Sai số (Gap %)** so với đích đến tối ưu $7542$.
---

## **2.1 Nearest Insertion**

In [16]:
def nearest_insertion(instance, start_node=0):
    n = len(instance.nodes)
    dist = instance.dist

    # 1. Khởi tạo: Bắt đầu tour với đỉnh xuất phát
    tour = [start_node]
    unvisited = set(range(n))
    unvisited.remove(start_node)

    # Mảng lưu khoảng cách ngắn nhất từ mỗi đỉnh chưa thăm tới CẢ TOUR hiện tại
    min_dist_to_tour = {v: dist[start_node][v] for v in unvisited}

    while unvisited:
        # Chọn đỉnh k chưa thăm có khoảng cách gần nhất với bất kỳ đỉnh nào trong tour
        best_k = min(unvisited, key=lambda v: min_dist_to_tour[v])

        if len(tour) == 1:
            best_idx = 1
        else:
            best_insertion_cost = float('inf')
            best_idx = -1
            m = len(tour)

            for i in range(m):
                u = tour[i]
                v = tour[(i + 1) % m] 

                # Chi phí tăng thêm (Delta) = d(u, k) + d(k, v) - d(u, v)
                cost = dist[u][best_k] + dist[best_k][v] - dist[u][v]
                if cost < best_insertion_cost:
                    best_insertion_cost = cost
                    best_idx = i + 1

       
        tour.insert(best_idx, best_k)
        unvisited.remove(best_k)

        for v in unvisited:
            if dist[best_k][v] < min_dist_to_tour[v]:
                min_dist_to_tour[v] = dist[best_k][v]

    # Tính tổng chi phí bằng hàm bạn đã định nghĩa
    total_cost = tour_length(tour, dist)

    return tour, total_cost

In [17]:
def swap_first_improvement(instance, initial_tour):
    dist = instance.dist
    n = len(initial_tour)
    tour = initial_tour[:]
    current_cost = tour_length(tour, dist)
    iterations = 0

    improved = True
    while improved:
        improved = False
        for i in range(n):
            for j in range(i + 1, n):
                # 1. Thử đổi chỗ
                tour[i], tour[j] = tour[j], tour[i]
                new_cost = tour_length(tour, dist)

                # 2. Nếu thực sự tốt hơn thì giữ lại
                if new_cost < current_cost:
                    current_cost = new_cost
                    iterations += 1
                    improved = True
                    break # Tìm thấy là nhảy ngay sang vòng lặp mới
                else:
                    # 3. Nếu không tốt hơn thì phải ĐỔI NGƯỢC LẠI
                    tour[i], tour[j] = tour[j], tour[i]

            if improved:
                break

    return tour, current_cost, iterations

In [18]:
def swap_best_improvement(instance, initial_tour):
    dist = instance.dist
    n = len(initial_tour)
    current_tour = initial_tour[:]
    current_cost = tour_length(current_tour, dist)

    iters = 0
    improved = True

    while improved:
        improved = False
        best_delta = 0
        best_pair = None

        for i in range(n):
            for j in range(i + 1, n):
                current_tour[i], current_tour[j] = current_tour[j], current_tour[i]
                new_cost = tour_length(current_tour, dist)

                delta = new_cost - current_cost

                if delta < best_delta:
                    best_delta = delta
                    best_pair = (i, j)

                # Trả lại vị trí cũ để thử cặp tiếp theo
                current_tour[i], current_tour[j] = current_tour[j], current_tour[i]

        if best_pair:
            i, j = best_pair
            current_tour[i], current_tour[j] = current_tour[j], current_tour[i]
            current_cost += best_delta
            iters += 1
            improved = True  

    return current_tour, current_cost, iters

In [19]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE  = r"Dataset/berlin52.tsp"
    OPTIMAL   = 7542

    # [1] Đọc dữ liệu 
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # [2] Chạy thuật toán khởi tạo
    t0_ni = time.perf_counter()
    ni_tour, ni_cost = nearest_insertion(instance, start_node=0)
    t_ni = (time.perf_counter() - t0_ni) * 1000

    # Silent Validation
    assert is_valid_tour(ni_tour, n)[0], "Lỗi: Tour khởi tạo không hợp lệ!"

    # Chạy Local Search variants
    ls_configs = [
        ("2opt-BI", two_opt_best_improvement),
        ("2opt-FI", two_opt_first_improvement),
        ("Swap-FI", swap_first_improvement),
        ("Swap-BI", swap_best_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, ni_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        # Silent Validation
        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [2] Summary Comparison (
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<30} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<30} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    # In kết quả gốc
    _print_row("Nearest Insertion (Base)", ni_cost, t_ni)

    # In kết quả Local Search
    for label, _, cost, t_ms in results:
        _print_row(f"NI + {label}", cost, t_ms)

    print("\n" + "=" * 68)
    print("PROCESS COMPLETE (All solutions validated)")
    print("=" * 68)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/berlin52.tsp...
    Cities (n): 52

[2] Summary Comparison (Optimal: 7,542):
  Method                             Distance    Time (ms)      Gap %
  -------------------------------------------------------------------
  Nearest Insertion (Base)              8,991         2.07     19.21%
  NI + 2opt-BI                          8,836         5.08     17.16%
  NI + 2opt-FI                          8,836         4.18     17.16%
  NI + Swap-FI                          8,941        50.02     18.55%
  NI + Swap-BI                          8,941        64.15     18.55%

PROCESS COMPLETE (All solutions validated)


## **2.2 Farthest Insertion**

In [20]:
def farthest_insertion(instance, start_node=0):
    dist = instance.dist
    n = len(instance.nodes)

    # 1. Khởi tạo
    unvisited = list(range(n))
    unvisited.remove(start_node)

    # Tìm thành phố xa start_node nhất để tạo cạnh đầu tiên
    farthest_node = max(unvisited, key=lambda x: dist[start_node][x])
    unvisited.remove(farthest_node)

    tour = [start_node, farthest_node]

    # Mảng lưu khoảng cách nhỏ nhất từ mỗi node chưa thăm đến tour hiện tại
    # Giúp giảm việc tính toán lặp lại
    min_dist_to_tour = [min(dist[i][start_node], dist[i][farthest_node]) for i in range(n)]

    while unvisited:
        # 2. SELECTION: Chọn node k nằm xa tour nhất
        # (Node có khoảng cách 'gần nhất' tới tour là lớn nhất)
        k = -1
        max_d = -1
        for node in unvisited:
            if min_dist_to_tour[node] > max_d:
                max_d = min_dist_to_tour[node]
                k = node

        unvisited.remove(k)

        # 3. INSERTION: Tìm vị trí (i, j) trong tour để chèn k vào sao cho
        # chi phí tăng thêm: d(i,k) + d(k,j) - d(i,j) là nhỏ nhất
        best_increase = float('inf')
        insertion_index = -1

        num_cities_in_tour = len(tour)
        for i in range(num_cities_in_tour):
            node_i = tour[i]
            node_j = tour[(i + 1) % num_cities_in_tour] # node kế tiếp (vòng tròn)

            increase = dist[node_i][k] + dist[k][node_j] - dist[node_i][node_j]
            if increase < best_increase:
                best_increase = increase
                insertion_index = i + 1

        # Chèn vào vị trí tốt nhất
        tour.insert(insertion_index, k)

        # 4. UPDATE: Cập nhật lại mảng min_dist_to_tour
        for node in unvisited:
            if dist[node][k] < min_dist_to_tour[node]:
                min_dist_to_tour[node] = dist[node][k]

    # Tính tổng quãng đường cuối cùng
    total_cost = sum(dist[tour[i]][tour[(i + 1) % n]] for i in range(n))

    return tour, total_cost

In [21]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE  = r"Dataset/berlin52.tsp"
    OPTIMAL   = 7542

    # [1] Đọc dữ liệu (Giữ lại để biết file đang xử lý)
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # [2] Chạy thuật toán khởi tạo
    t0_ni = time.perf_counter()
    fi_tour, fi_cost = farthest_insertion(instance, start_node=0)
    t_ni = (time.perf_counter() - t0_ni) * 1000

    # Silent Validation
    assert is_valid_tour(fi_tour, n)[0], "Lỗi: Tour khởi tạo không hợp lệ!"

    # Chạy Local Search variants
    ls_configs = [
        ("2opt-BI", two_opt_best_improvement),
        ("2opt-FI", two_opt_first_improvement),
        ("Swap-FI", swap_first_improvement),
        ("Swap-BI", swap_best_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, fi_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        # Silent Validation
        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [2] Summary Comparison (
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<30} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<30} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    # In kết quả gốc
    _print_row("Farthest Insertion (Base)", fi_cost, t_ni)

    # In kết quả Local Search
    for label, _, cost, t_ms in results:
        _print_row(f"FI + {label}", cost, t_ms)

    print("\n" + "=" * 68)
    print("PROCESS COMPLETE (All solutions validated)")
    print("=" * 68)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/berlin52.tsp...
    Cities (n): 52

[2] Summary Comparison (Optimal: 7,542):
  Method                             Distance    Time (ms)      Gap %
  -------------------------------------------------------------------
  Farthest Insertion (Base)             8,307         1.81     10.14%
  FI + 2opt-BI                          8,194         5.56      8.64%
  FI + 2opt-FI                          8,194         3.83      8.64%
  FI + Swap-FI                          8,194        53.60      8.64%
  FI + Swap-BI                          8,194        39.67      8.64%

PROCESS COMPLETE (All solutions validated)


## **2.3 Cheapest Insertion**

In [22]:
def cheapest_insertion(instance, start_node=0):
    dist = instance.dist
    n = len(instance.nodes)

    # 1. Khởi tạo
    unvisited = list(range(n))
    unvisited.remove(start_node)

    # Tìm node gần nhất với start_node để tạo tour ban đầu 
    first_neighbor = min(unvisited, key=lambda x: dist[start_node][x])
    unvisited.remove(first_neighbor)

    tour = [start_node, first_neighbor]

    # Duy trì chi phí chèn tốt nhất cho mỗi node chưa thăm
    # Cấu trúc: min_insertion_info[node] = (chi phí tăng thêm, vị trí chèn)
    def get_best_insertion_for_node(node, current_tour):
        best_inc = float('inf')
        best_idx = -1
        for i in range(len(current_tour)):
            u = current_tour[i]
            v = current_tour[(i + 1) % len(current_tour)]
            # Công cụ tính chi phí tăng thêm: d(u,k) + d(k,v) - d(u,v)
            increase = dist[u][node] + dist[node][v] - dist[u][v]
            if increase < best_inc:
                best_inc = increase
                best_idx = i + 1
        return best_inc, best_idx

    # Khởi tạo mảng thông tin chèn
    insertion_info = {node: get_best_insertion_for_node(node, tour) for node in unvisited}

    while unvisited:
        # 2. SELECTION: Chọn node k có chi phí chèn "rẻ" nhất toàn cục
        k = min(unvisited, key=lambda x: insertion_info[x][0])
        cost_inc, idx = insertion_info[k]

        # 3. INSERTION: Chèn k vào tour tại vị trí đã tìm thấy
        tour.insert(idx, k)
        unvisited.remove(k)
        del insertion_info[k]

        # 4. UPDATE: Cập nhật lại insertion_info cho các node còn lại
        # Chỉ cần kiểm tra xem việc chèn k có tạo ra vị trí chèn mới tốt hơn không
        for node in unvisited:
            insertion_info[node] = get_best_insertion_for_node(node, tour)

    # Tính tổng quãng đường
    total_cost = sum(dist[tour[i]][tour[(i + 1) % n]] for i in range(n))

    return tour, total_cost

In [23]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE  = r"Dataset/berlin52.tsp"
    OPTIMAL   = 7542

    # [1] Đọc dữ liệu 
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # [2] Chạy thuật toán khởi tạo
    t0_ni = time.perf_counter()
    ci_tour, ci_cost = cheapest_insertion(instance, start_node=0)
    t_ni = (time.perf_counter() - t0_ni) * 1000

    # Silent Validation
    assert is_valid_tour(ci_tour, n)[0], "Lỗi: Tour khởi tạo không hợp lệ!"

    # Chạy Local Search variants
    ls_configs = [
        ("2opt-BI", two_opt_best_improvement),
        ("2opt-FI", two_opt_first_improvement),
        ("Swap-FI", swap_first_improvement),
        ("Swap-BI", swap_best_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, ci_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        # Silent Validation
        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [2] Summary Comparison (
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<30} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<30} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    # In kết quả gốc
    _print_row("Cheapest Insertion (Base)", ci_cost, t_ni)

    # In kết quả Local Search
    for label, _, cost, t_ms in results:
        _print_row(f"CI + {label}", cost, t_ms)

    print("\n" + "=" * 68)
    print("PROCESS COMPLETE (All solutions validated)")
    print("=" * 68)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/berlin52.tsp...
    Cities (n): 52

[2] Summary Comparison (Optimal: 7,542):
  Method                             Distance    Time (ms)      Gap %
  -------------------------------------------------------------------
  Cheapest Insertion (Base)             9,004        24.84     19.38%
  CI + 2opt-BI                          8,542        14.18     13.26%
  CI + 2opt-FI                          8,396         4.77     11.32%
  CI + Swap-FI                          8,865        39.36     17.54%
  CI + Swap-BI                          8,865       113.59     17.54%

PROCESS COMPLETE (All solutions validated)


## **2.4 Greedy Edge**

In [24]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
    def find(self, i):
        if self.parent[i] == i: return i
        self.parent[i] = self.find(self.parent[i])
        return self.parent[i]
    def union(self, i, j):
        root_i = self.find(i)
        root_j = self.find(j)
        if root_i != root_j:
            self.parent[root_i] = root_j
            return True
        return False

def greedy_edge(instance):
    dist = instance.dist
    n = len(instance.nodes)

    # 1. Tạo danh sách tất cả các cạnh và sắp xếp tăng dần theo độ dài
    edges = []
    for i in range(n):
        for j in range(i + 1, n):
            edges.append((dist[i][j], i, j))
    edges.sort() # O(E log E) -> O(n^2 log n)

    adj = [[] for _ in range(n)]
    degrees = [0] * n
    dsu = DSU(n)
    edges_count = 0
    total_cost = 0

    # 2. Duyệt qua các cạnh đã sắp xếp
    for d, u, v in edges:
        if edges_count == n - 1: break # Đã đủ n-1 cạnh, cạnh cuối sẽ nối để đóng vòng

        # Điều kiện chọn cạnh:
        # - Bậc của mỗi đỉnh không được vượt quá 2
        # - Không tạo thành chu trình con (trừ khi là cạnh cuối cùng)
        if degrees[u] < 2 and degrees[v] < 2:
            if dsu.find(u) != dsu.find(v):
                dsu.union(u, v)
                adj[u].append(v)
                adj[v].append(u)
                degrees[u] += 1
                degrees[v] += 1
                edges_count += 1
                total_cost += d

    # 3. Nối cạnh cuối cùng để đóng kín chu trình
    last_nodes = [i for i in range(n) if degrees[i] == 1]
    if len(last_nodes) == 2:
        u, v = last_nodes
        adj[u].append(v)
        adj[v].append(u)
        total_cost += dist[u][v]

    # 4. Chuyển đổi danh sách kề (adj) thành một Tour (list)
    tour = []
    curr = 0
    prev = -1
    for _ in range(n):
        tour.append(curr)
        next_node = adj[curr][0] if adj[curr][0] != prev else adj[curr][1]
        prev, curr = curr, next_node

    return tour, total_cost

In [25]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE  = r"Dataset/berlin52.tsp"
    OPTIMAL   = 7542

    # [1] Đọc dữ liệu
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # [2] Chạy thuật toán khởi tạo: Greedy Edge (Chạy ngầm)
    t0_ge = time.perf_counter()
    ge_tour, ge_cost = greedy_edge(instance)
    t_ge = (time.perf_counter() - t0_ge) * 1000

    # Silent Validation
    assert is_valid_tour(ge_tour, n)[0], "Lỗi: Tour khởi tạo Greedy Edge không hợp lệ!"

    # [3] Chạy Local Search variants (Chạy ngầm)
    ls_configs = [
        ("2opt-FI", two_opt_first_improvement),
        ("2opt-BI", two_opt_best_improvement),
        ("Swap-FI", swap_first_improvement),
        ("Swap-BI", swap_best_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, ge_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        # Silent Validation
        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [4] Summary Comparison
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<30} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<30} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    # In kết quả gốc
    _print_row("Greedy Edge (Base)", ge_cost, t_ge)

    # Sắp xếp kết quả LS theo quãng đường tăng dần
    results.sort(key=lambda x: x[2])
    for label, _, cost, t_ms in results:
        _print_row(f"GE + {label}", cost, t_ms)

    # In thông tin lộ trình tốt nhất (Tùy chọn giữ lại để check kết quả cụ thể)
    best_label, best_tour, best_cost, _ = results[0]
    print(f"\n[3] Best Route Found: GE + {best_label} ({best_cost:,})")

    print("\n" + "=" * 68)
    print("PROCESS COMPLETE (All solutions validated)")
    print("=" * 68)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/berlin52.tsp...
    Cities (n): 52

[2] Summary Comparison (Optimal: 7,542):
  Method                             Distance    Time (ms)      Gap %
  -------------------------------------------------------------------
  Greedy Edge (Base)                    9,951         2.17     31.94%
  GE + 2opt-FI                          8,156        11.20      8.14%
  GE + 2opt-BI                          8,297        23.85     10.01%
  GE + Swap-BI                          8,933       224.37     18.44%
  GE + Swap-FI                          9,285       139.51     23.11%

[3] Best Route Found: GE + 2opt-FI (8,156)

PROCESS COMPLETE (All solutions validated)


# **II. DATASET: KROA100**

# BỘ DỮ LIỆU CHUẨN: KROA100 (TSPLIB)

> **Giới thiệu:** `kroA100` là một bộ dữ liệu kiểm thử nổi tiếng khác thuộc thư viện **TSPLIB**. Bộ dữ liệu này chứa tọa độ 2D của **100 thành phố** (được mô phỏng ngẫu nhiên tại các vị trí địa lý thuộc nước Mỹ bởi tác giả Krolak). Đây là "bài kiểm tra hạng nặng" tiếp theo để chúng ta so sánh xem các thuật toán Greedy hay Local Search sẽ xoay sở ra sao khi quy mô bài toán tăng lên gấp đôi so với `berlin52`.

---

### Thông Số "Hồ Sơ" Của Dataset

Để dễ lập trình và cấu hình, dưới đây là các thông tin cốt lõi của bộ dữ liệu này:

| Thuộc tính | Giá trị chi tiết | Ý nghĩa |
| :--- | :--- | :--- |
| **Name** | `kroA100` | Tên định danh trong thư viện TSPLIB |
| **Type** | `TSP` | Bài toán Người giao hàng đối xứng (Symmetric TSP) |
| **Dimension** | `100` | **Tổng số lượng thành phố (100 nút đồ thị)** |
| **Edge Weight Type** | `EUC_2D` | Khoảng cách Euclid phẳng (Hệ tọa độ 2D) |
| **Best Known Optimal** | $\mathbf{21282}$ | **Độ dài quãng đường ngắn nhất tối ưu** |

---

###  Công Thức Tính Khoảng Cách

Tương tự như các bài toán TSP phẳng khác, khoảng cách giữa thành phố $i$ và thành phố $j$ được tính bằng công thức Euclid và làm tròn thành số nguyên:

$$d(i, j) = \text{round}\left( \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2} \right)$$

---


## **1. RNN + RELOCATE**

In [26]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE = r"Dataset/kroA100.tsp"
    OPTIMAL  = 21282

    # [1] Đọc dữ liệu
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # Khởi tạo tour: Repeated Nearest Neighbor (Chạy ngầm)
    t0_rnn = time.perf_counter()
    rnn_tour, rnn_cost, best_start = repeated_nearest_neighbor(instance)
    t_rnn = (time.perf_counter() - t0_rnn) * 1000

    # Silent Validation
    assert is_valid_tour(rnn_tour, n)[0], "Lỗi: Tour RNN không hợp lệ!"

    # Chạy Local Search variants (Chạy ngầm)
    ls_configs = [
        ("Relocate-BI", relocate_best_improvement),
        ("Relocate-FI", relocate_first_improvement),
        ("2opt-BI",     two_opt_best_improvement),
        ("2opt-FI",     two_opt_first_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, rnn_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        # Silent Validation
        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [2] Bảng so sánh tổng hợp (Summary Comparison)
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<35} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<35} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    # In kết quả gốc
    _print_row("Repeated Nearest Neighbor (RNN)", rnn_cost, t_rnn)

    # Sắp xếp kết quả LS theo quãng đường tăng dần để dễ so sánh
    results.sort(key=lambda x: x[2])
    for label, _, cost, t_ms in results:
        _print_row(f"RNN + {label}", cost, t_ms)

    # Lưu dữ liệu vào biến global để cell Visualization sử dụng
    global viz_data
    viz_data = {
        'instance': instance,
        'rnn_tour': rnn_tour,
        'rnn_cost': rnn_cost,
        'best_start': best_start,
        'results': results
    }

    best_label, _, best_cost, _ = results[0]
    print(f"\n[3] Best Solution: RNN + {best_label} ({best_cost:,})")
    print("\n" + "=" * 70)
    print("PROCESS COMPLETE (All solutions validated silently)")
    print("=" * 70)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/kroA100.tsp...
    Cities (n): 100

[2] Summary Comparison (Optimal: 21,282):
  Method                                  Distance    Time (ms)      Gap %
  ------------------------------------------------------------------------
  Repeated Nearest Neighbor (RNN)           24,698       172.08     16.05%
  RNN + 2opt-BI                             21,389        79.07      0.50%
  RNN + 2opt-FI                             21,807        81.98      2.47%
  RNN + Relocate-BI                         21,904       128.37      2.92%
  RNN + Relocate-FI                         23,147        72.18      8.76%

[3] Best Solution: RNN + 2opt-BI (21,389)

PROCESS COMPLETE (All solutions validated silently)


## **2. Nearest Insertion**

In [27]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE = r"Dataset/kroA100.tsp"  
    OPTIMAL  = 21282          

    # [1] Đọc dữ liệu (Giữ lại để biết file đang xử lý)
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # [2] Chạy thuật toán khởi tạo
    t0_ni = time.perf_counter()
    ni_tour, ni_cost = nearest_insertion(instance, start_node=0)
    t_ni = (time.perf_counter() - t0_ni) * 1000

    # Silent Validation
    assert is_valid_tour(ni_tour, n)[0], "Lỗi: Tour khởi tạo không hợp lệ!"

    # Chạy Local Search variants
    ls_configs = [
        ("2opt-BI", two_opt_best_improvement),
        ("2opt-FI", two_opt_first_improvement),
        ("Swap-FI", swap_first_improvement),
        ("Swap-BI", swap_best_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, ni_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        # Silent Validation
        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [2] Summary Comparison (
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<30} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<30} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    # In kết quả gốc
    _print_row("Nearest Insertion (Base)", ni_cost, t_ni)

    # In kết quả Local Search
    for label, _, cost, t_ms in results:
        _print_row(f"NI + {label}", cost, t_ms)

    print("\n" + "=" * 68)
    print("PROCESS COMPLETE (All solutions validated)")
    print("=" * 68)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/kroA100.tsp...
    Cities (n): 100

[2] Summary Comparison (Optimal: 21,282):
  Method                             Distance    Time (ms)      Gap %
  -------------------------------------------------------------------
  Nearest Insertion (Base)             25,868         9.50     21.55%
  NI + 2opt-BI                         22,681        78.44      6.57%
  NI + 2opt-FI                         23,007        45.52      8.11%
  NI + Swap-FI                         24,471      1936.80     14.98%
  NI + Swap-BI                         24,612      2037.40     15.65%

PROCESS COMPLETE (All solutions validated)


## **3. Farthest Insertion**

In [28]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE = r"Dataset/kroA100.tsp"
    OPTIMAL  = 21282

    # [1] Đọc dữ liệu
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # [2] Chạy thuật toán khởi tạo
    t0_ni = time.perf_counter()
    fi_tour, fi_cost = farthest_insertion(instance, start_node=0)
    t_ni = (time.perf_counter() - t0_ni) * 1000

    # Silent Validation
    assert is_valid_tour(fi_tour, n)[0], "Lỗi: Tour khởi tạo không hợp lệ!"

    # Chạy Local Search variants
    ls_configs = [
        ("2opt-BI", two_opt_best_improvement),
        ("2opt-FI", two_opt_first_improvement),
        ("Swap-FI", swap_first_improvement),
        ("Swap-BI", swap_best_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, fi_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        # Silent Validation
        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [2] Summary Comparison (
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<30} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<30} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    # In kết quả gốc
    _print_row("Farthest Insertion (Base)", fi_cost, t_ni)

    # In kết quả Local Search
    for label, _, cost, t_ms in results:
        _print_row(f"FI + {label}", cost, t_ms)

    print("\n" + "=" * 68)
    print("PROCESS COMPLETE (All solutions validated)")
    print("=" * 68)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/kroA100.tsp...
    Cities (n): 100

[2] Summary Comparison (Optimal: 21,282):
  Method                             Distance    Time (ms)      Gap %
  -------------------------------------------------------------------
  Farthest Insertion (Base)            23,186         6.31      8.95%
  FI + 2opt-BI                         22,855        29.42      7.39%
  FI + 2opt-FI                         22,855        65.30      7.39%
  FI + Swap-FI                         23,057       682.71      8.34%
  FI + Swap-BI                         23,057       380.04      8.34%

PROCESS COMPLETE (All solutions validated)


## **4. Cheapest Inserttion**

In [29]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE = r"Dataset/kroA100.tsp"
    OPTIMAL  = 21282

    # [1] Đọc dữ liệu
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # [2] Chạy thuật toán khởi tạo
    t0_ni = time.perf_counter()
    ci_tour, ci_cost = cheapest_insertion(instance, start_node=0)
    t_ni = (time.perf_counter() - t0_ni) * 1000

    # Silent Validation
    assert is_valid_tour(ci_tour, n)[0], "Lỗi: Tour khởi tạo không hợp lệ!"

    # Chạy Local Search variants
    ls_configs = [
        ("2opt-BI", two_opt_best_improvement),
        ("2opt-FI", two_opt_first_improvement),
        ("Swap-FI", swap_first_improvement),
        ("Swap-BI", swap_best_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, ci_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        # Silent Validation
        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [2] Summary Comparison (
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<30} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<30} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    # In kết quả gốc
    _print_row("Cheapest Insertion (Base)", ci_cost, t_ni)

    # In kết quả Local Search
    for label, _, cost, t_ms in results:
        _print_row(f"CI + {label}", cost, t_ms)

    print("\n" + "=" * 68)
    print("PROCESS COMPLETE (All solutions validated)")
    print("=" * 68)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/kroA100.tsp...
    Cities (n): 100

[2] Summary Comparison (Optimal: 21,282):
  Method                             Distance    Time (ms)      Gap %
  -------------------------------------------------------------------
  Cheapest Insertion (Base)            24,942       172.53     17.20%
  CI + 2opt-BI                         23,806        49.64     11.86%
  CI + 2opt-FI                         23,806        46.82     11.86%
  CI + Swap-FI                         23,806      1855.31     11.86%
  CI + Swap-BI                         23,806      1478.69     11.86%

PROCESS COMPLETE (All solutions validated)


## **5. Greedy Edge**

In [30]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE = r"Dataset/kroA100.tsp"
    OPTIMAL  = 21282

    # [1] Đọc dữ liệu
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # [2] Chạy thuật toán khởi tạo: Greedy Edge (Chạy ngầm)
    t0_ge = time.perf_counter()
    ge_tour, ge_cost = greedy_edge(instance)
    t_ge = (time.perf_counter() - t0_ge) * 1000

    # Silent Validation
    assert is_valid_tour(ge_tour, n)[0], "Lỗi: Tour khởi tạo Greedy Edge không hợp lệ!"

    # [3] Chạy Local Search variants (Chạy ngầm)
    ls_configs = [
        ("2opt-FI", two_opt_first_improvement),
        ("2opt-BI", two_opt_best_improvement),
        ("Swap-FI", swap_first_improvement),
        ("Swap-BI", swap_best_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, ge_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        # Silent Validation
        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [4] Summary Comparison
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<30} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<30} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    # In kết quả gốc
    _print_row("Greedy Edge (Base)", ge_cost, t_ge)

    # Sắp xếp kết quả LS theo quãng đường tăng dần
    results.sort(key=lambda x: x[2])
    for label, _, cost, t_ms in results:
        _print_row(f"GE + {label}", cost, t_ms)

    # In thông tin lộ trình tốt nhất (Tùy chọn giữ lại để check kết quả cụ thể)
    best_label, best_tour, best_cost, _ = results[0]
    print(f"\n[3] Best Route Found: GE + {best_label} ({best_cost:,})")

    print("\n" + "=" * 68)
    print("PROCESS COMPLETE (All solutions validated)")
    print("=" * 68)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/kroA100.tsp...
    Cities (n): 100

[2] Summary Comparison (Optimal: 21,282):
  Method                             Distance    Time (ms)      Gap %
  -------------------------------------------------------------------
  Greedy Edge (Base)                   24,287         6.72     14.12%
  GE + 2opt-BI                         21,393        67.51      0.52%
  GE + 2opt-FI                         21,834        80.68      2.59%
  GE + Swap-FI                         22,813      1113.47      7.19%
  GE + Swap-BI                         22,813      1496.66      7.19%

[3] Best Route Found: GE + 2opt-BI (21,393)

PROCESS COMPLETE (All solutions validated)


# **III. DATASET: CH130**
#  BỘ DỮ LIỆU CHUẨN: CH130 (TSPLIB)

> **Giới thiệu:** `ch130` là bộ dữ liệu chuẩn mực đại diện cho mạng lưới gồm **130 thành phố tại Thụy Sĩ (Switzerland)**. Sau khi đã thử sức với 52 thành phố (`berlin52`) và 100 thành phố (`kroA100`), đây chính là "vòng dốc" tiếp theo để chúng ta thử thách các thuật toán. Quy mô dữ liệu lớn hơn đồng nghĩa với việc các thuật toán tồi sẽ rất dễ bị "rùa bò" (tốn thời gian) hoặc bị kẹt lại ở những đường đi rất xấu.

---

### Thông Số "Hồ Sơ" Của Dataset

Dưới đây là các thông tin cấu hình cốt lõi của "đấu trường" 130 thành phố này:

| Thuộc tính | Giá trị chi tiết | Ý nghĩa |
| :--- | :--- | :--- |
| **Name** | `ch130` | Tên định danh trong thư viện TSPLIB |
| **Type** | `TSP` | Bài toán Người giao hàng đối xứng (Symmetric TSP) |
| **Dimension** | `130` | **Tổng số lượng thành phố (130 nút đồ thị)** |
| **Edge Weight Type** | `EUC_2D` | Khoảng cách Euclid phẳng (Hệ tọa độ 2D) |
| **Best Known Optimal** | $\mathbf{6110}$ | **Độ dài quãng đường ngắn nhất tối ưu** |

---

### Cách Tính Khoảng Cách

Vẫn tuân thủ theo luật chơi chung của thư viện TSPLIB, khoảng cách giữa các thành phố được tính bằng công thức Euclid trên mặt phẳng 2D và làm tròn về số nguyên gần nhất:

$$d(i, j) = \text{round}\left( \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2} \right)$$

---



## **1. Repeated Nearest Neighborhood**


In [31]:
import time

# --- Định dạng tiêu đề to bằng Markdown ---
# <h1 style="font-size:30px; font-weight:bold;">1. TSP SOLVER EXECUTION</h1>

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE = r"Dataset/ch130.tsp" 
    OPTIMAL  = 6110           

    # [1] Đọc dữ liệu
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # Khởi tạo tour: Repeated Nearest Neighbor (Chạy ngầm)
    t0_rnn = time.perf_counter()
    rnn_tour, rnn_cost, best_start = repeated_nearest_neighbor(instance)
    t_rnn = (time.perf_counter() - t0_rnn) * 1000

    # Silent Validation
    assert is_valid_tour(rnn_tour, n)[0], "Lỗi: Tour RNN không hợp lệ!"

    # Chạy Local Search variants (Chạy ngầm)
    ls_configs = [
        ("Relocate-BI", relocate_best_improvement),
        ("Relocate-FI", relocate_first_improvement),
        ("2opt-BI",     two_opt_best_improvement),
        ("2opt-FI",     two_opt_first_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, rnn_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        # Silent Validation
        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [2] Bảng so sánh tổng hợp (Summary Comparison)
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<35} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<35} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    # In kết quả gốc
    _print_row("Repeated Nearest Neighbor (RNN)", rnn_cost, t_rnn)

    # Sắp xếp kết quả LS theo quãng đường tăng dần để dễ so sánh
    results.sort(key=lambda x: x[2])
    for label, _, cost, t_ms in results:
        _print_row(f"RNN + {label}", cost, t_ms)

    # Lưu dữ liệu vào biến global để cell Visualization sử dụng
    global viz_data
    viz_data = {
        'instance': instance,
        'rnn_tour': rnn_tour,
        'rnn_cost': rnn_cost,
        'best_start': best_start,
        'results': results
    }

    best_label, _, best_cost, _ = results[0]
    print(f"\n[3] Best Solution: RNN + {best_label} ({best_cost:,})")
    print("\n" + "=" * 70)
    print("PROCESS COMPLETE (All solutions validated silently)")
    print("=" * 70)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/ch130.tsp...
    Cities (n): 130

[2] Summary Comparison (Optimal: 6,110):
  Method                                  Distance    Time (ms)      Gap %
  ------------------------------------------------------------------------
  Repeated Nearest Neighbor (RNN)            7,129       225.87     16.68%
  RNN + 2opt-FI                              6,404       131.77      4.81%
  RNN + 2opt-BI                              6,414       157.71      4.98%
  RNN + Relocate-BI                          6,737       227.13     10.26%
  RNN + Relocate-FI                          6,762       159.33     10.67%

[3] Best Solution: RNN + 2opt-FI (6,404)

PROCESS COMPLETE (All solutions validated silently)


# **2. Nearest Insertion**

In [32]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE = r"Dataset/ch130.tsp"
    OPTIMAL  =  6110

    # [1] Đọc dữ liệu (Giữ lại để biết file đang xử lý)
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # [2] Chạy thuật toán khởi tạo
    t0_ni = time.perf_counter()
    ni_tour, ni_cost = nearest_insertion(instance, start_node=0)
    t_ni = (time.perf_counter() - t0_ni) * 1000

    # Silent Validation
    assert is_valid_tour(ni_tour, n)[0], "Lỗi: Tour khởi tạo không hợp lệ!"

    # Chạy Local Search variants
    ls_configs = [
        ("2opt-BI", two_opt_best_improvement),
        ("2opt-FI", two_opt_first_improvement),
        ("Swap-FI", swap_first_improvement),
        ("Swap-BI", swap_best_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, ni_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        # Silent Validation
        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [2] Summary Comparison (
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<30} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<30} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    # In kết quả gốc
    _print_row("Nearest Insertion (Base)", ni_cost, t_ni)

    # In kết quả Local Search
    for label, _, cost, t_ms in results:
        _print_row(f"NI + {label}", cost, t_ms)

    print("\n" + "=" * 68)
    print("PROCESS COMPLETE (All solutions validated)")
    print("=" * 68)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/ch130.tsp...
    Cities (n): 130

[2] Summary Comparison (Optimal: 6,110):
  Method                             Distance    Time (ms)      Gap %
  -------------------------------------------------------------------
  Nearest Insertion (Base)              7,446        12.25     21.87%
  NI + 2opt-BI                          6,486       222.30      6.15%
  NI + 2opt-FI                          6,508       261.64      6.51%
  NI + Swap-FI                          6,969      3756.11     14.06%
  NI + Swap-BI                          7,005      4987.08     14.65%

PROCESS COMPLETE (All solutions validated)


# **3.Farthest Insertion**

In [33]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE = r"Dataset/ch130.tsp"
    OPTIMAL  =  6110

    # [1] Đọc dữ liệu (Giữ lại để biết file đang xử lý)
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # [2] Chạy thuật toán khởi tạo
    t0_ni = time.perf_counter()
    fi_tour, fi_cost = farthest_insertion(instance, start_node=0)
    t_ni = (time.perf_counter() - t0_ni) * 1000

    # Silent Validation
    assert is_valid_tour(fi_tour, n)[0], "Lỗi: Tour khởi tạo không hợp lệ!"

    # Chạy Local Search variants
    ls_configs = [
        ("2opt-BI", two_opt_best_improvement),
        ("2opt-FI", two_opt_first_improvement),
        ("Swap-FI", swap_first_improvement),
        ("Swap-BI", swap_best_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, fi_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        # Silent Validation
        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [2] Summary Comparison (
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<30} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<30} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    # In kết quả gốc
    _print_row("Farthest Insertion (Base)", fi_cost, t_ni)

    # In kết quả Local Search
    for label, _, cost, t_ms in results:
        _print_row(f"FI + {label}", cost, t_ms)

    print("\n" + "=" * 68)
    print("PROCESS COMPLETE (All solutions validated)")
    print("=" * 68)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/ch130.tsp...
    Cities (n): 130

[2] Summary Comparison (Optimal: 6,110):
  Method                             Distance    Time (ms)      Gap %
  -------------------------------------------------------------------
  Farthest Insertion (Base)             6,433         7.85      5.29%
  FI + 2opt-BI                          6,388        25.83      4.55%
  FI + 2opt-FI                          6,388        14.72      4.55%
  FI + Swap-FI                          6,388       824.08      4.55%
  FI + Swap-BI                          6,388       933.90      4.55%

PROCESS COMPLETE (All solutions validated)


# **4.Cheapest Insertion**

In [34]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE = r"Dataset/ch130.tsp"
    OPTIMAL  =  6110

    # [1] Đọc dữ liệu (Giữ lại để biết file đang xử lý)
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # [2] Chạy thuật toán khởi tạo
    t0_ni = time.perf_counter()
    ci_tour, ci_cost = cheapest_insertion(instance, start_node=0)
    t_ni = (time.perf_counter() - t0_ni) * 1000

    # Silent Validation
    assert is_valid_tour(ci_tour, n)[0], "Lỗi: Tour khởi tạo không hợp lệ!"

    # Chạy Local Search variants
    ls_configs = [
        ("2opt-BI", two_opt_best_improvement),
        ("2opt-FI", two_opt_first_improvement),
        ("Swap-FI", swap_first_improvement),
        ("Swap-BI", swap_best_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, ci_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        # Silent Validation
        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [2] Summary Comparison (
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<30} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<30} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    # In kết quả gốc
    _print_row("Cheapest Insertion (Base)", ci_cost, t_ni)

    # In kết quả Local Search
    for label, _, cost, t_ms in results:
        _print_row(f"CI + {label}", cost, t_ms)

    print("\n" + "=" * 68)
    print("PROCESS COMPLETE (All solutions validated)")
    print("=" * 68)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/ch130.tsp...
    Cities (n): 130

[2] Summary Comparison (Optimal: 6,110):
  Method                             Distance    Time (ms)      Gap %
  -------------------------------------------------------------------
  Cheapest Insertion (Base)             7,301       212.95     19.49%
  CI + 2opt-BI                          6,705       104.22      9.74%
  CI + 2opt-FI                          6,683        76.05      9.38%
  CI + Swap-FI                          6,723      3704.29     10.03%
  CI + Swap-BI                          6,723      4736.12     10.03%

PROCESS COMPLETE (All solutions validated)


# **5. Greedy Edge**

In [35]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE = r"Dataset/ch130.tsp"
    OPTIMAL  =  6110
    # [1] Đọc dữ liệu
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # [2] Chạy thuật toán khởi tạo: Greedy Edge (Chạy ngầm)
    t0_ge = time.perf_counter()
    ge_tour, ge_cost = greedy_edge(instance)
    t_ge = (time.perf_counter() - t0_ge) * 1000

    # Silent Validation
    assert is_valid_tour(ge_tour, n)[0], "Lỗi: Tour khởi tạo Greedy Edge không hợp lệ!"

    # [3] Chạy Local Search variants (Chạy ngầm)
    ls_configs = [
        ("2opt-FI", two_opt_first_improvement),
        ("2opt-BI", two_opt_best_improvement),
        ("Swap-FI", swap_first_improvement),
        ("Swap-BI", swap_best_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, ge_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        # Silent Validation
        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [4] Summary Comparison
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<30} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<30} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    # In kết quả gốc
    _print_row("Greedy Edge (Base)", ge_cost, t_ge)

    # Sắp xếp kết quả LS theo quãng đường tăng dần
    results.sort(key=lambda x: x[2])
    for label, _, cost, t_ms in results:
        _print_row(f"GE + {label}", cost, t_ms)

    # In thông tin lộ trình tốt nhất (Tùy chọn giữ lại để check kết quả cụ thể)
    best_label, best_tour, best_cost, _ = results[0]
    print(f"\n[3] Best Route Found: GE + {best_label} ({best_cost:,})")

    print("\n" + "=" * 68)
    print("PROCESS COMPLETE (All solutions validated)")
    print("=" * 68)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/ch130.tsp...
    Cities (n): 130

[2] Summary Comparison (Optimal: 6,110):
  Method                             Distance    Time (ms)      Gap %
  -------------------------------------------------------------------
  Greedy Edge (Base)                    7,223        17.79     18.22%
  GE + 2opt-BI                          6,249       136.64      2.27%
  GE + 2opt-FI                          6,441       162.93      5.42%
  GE + Swap-BI                          7,020      3230.62     14.89%
  GE + Swap-FI                          7,114      1787.29     16.43%

[3] Best Route Found: GE + 2opt-BI (6,249)

PROCESS COMPLETE (All solutions validated)


# **IV. DATASET: EIL76**
#  BỘ DỮ LIỆU CHUẨN: EIL76 (TSPLIB)

> **Giới thiệu:** `eil76` là bộ dữ liệu kiểm thử thuộc thư viện **TSPLIB** (Đại học Heidelberg), bao gồm **76 thành phố** trong một mặt phẳng 2D (Euclidean). Được thiết kế bởi Christofides & Eilon, đây là benchmark tiêu chuẩn để đánh giá hiệu suất thuật toán TSP.

| Thuộc tính | Giá trị |
|---|---|
| Số thành phố (n) | 76 |
| Loại khoảng cách | EUC\_2D (Euclidean 2D) |
| Nghiệm tối ưu | **538** |
| Nguồn | TSPLIB — Christofides/Eilon |

## **1. Repeated Nearest Neighbor + Relocate**

In [36]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE = r"Dataset/eil76.tsp"
    OPTIMAL  = 538

    # [1] Đọc dữ liệu
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # Khởi tạo tour: Repeated Nearest Neighbor (Chạy ngầm)
    t0_rnn = time.perf_counter()
    rnn_tour, rnn_cost, best_start = repeated_nearest_neighbor(instance)
    t_rnn = (time.perf_counter() - t0_rnn) * 1000

    # Validation
    assert is_valid_tour(rnn_tour, n)[0], "Lỗi: Tour RNN không hợp lệ!"

    # Chạy Local Search variants (Chạy ngầm)
    ls_configs = [
        ("Relocate-BI", relocate_best_improvement),
        ("Relocate-FI", relocate_first_improvement),
        ("2opt-BI",     two_opt_best_improvement),
        ("2opt-FI",     two_opt_first_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, rnn_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [2] Bảng so sánh tổng hợp (Summary Comparison)
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<35} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<35} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    _print_row("Repeated Nearest Neighbor (RNN)", rnn_cost, t_rnn)

    results.sort(key=lambda x: x[2])
    for label, _, cost, t_ms in results:
        _print_row(f"RNN + {label}", cost, t_ms)

    best_label, _, best_cost, _ = results[0]
    print(f"\n[3] Best Solution: RNN + {best_label} ({best_cost:,})")
    print("\n" + "=" * 70)
    print("PROCESS COMPLETE (All solutions validated silently)")
    print("=" * 70)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/eil76.tsp...
    Cities (n): 76

[2] Summary Comparison (Optimal: 538):
  Method                                  Distance    Time (ms)      Gap %
  ------------------------------------------------------------------------
  Repeated Nearest Neighbor (RNN)              608        38.41     13.01%
  RNN + 2opt-BI                                554        21.24      2.97%
  RNN + 2opt-FI                                562        16.52      4.46%
  RNN + Relocate-BI                            566        36.47      5.20%
  RNN + Relocate-FI                            574        20.48      6.69%

[3] Best Solution: RNN + 2opt-BI (554)

PROCESS COMPLETE (All solutions validated silently)


## **2. Nearest Insertion**

In [37]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE = r"Dataset/eil76.tsp"
    OPTIMAL  = 538

    # [1] Đọc dữ liệu
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # [2] Chạy thuật toán khởi tạo
    t0_ni = time.perf_counter()
    ni_tour, ni_cost = nearest_insertion(instance, start_node=0)
    t_ni = (time.perf_counter() - t0_ni) * 1000

    assert is_valid_tour(ni_tour, n)[0], "Lỗi: Tour khởi tạo không hợp lệ!"

    # Chạy Local Search variants
    ls_configs = [
        ("2opt-BI", two_opt_best_improvement),
        ("2opt-FI", two_opt_first_improvement),
        ("Swap-FI", swap_first_improvement),
        ("Swap-BI", swap_best_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, ni_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [3] Summary Comparison
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<30} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<30} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    _print_row("Nearest Insertion (Base)", ni_cost, t_ni)

    for label, _, cost, t_ms in results:
        _print_row(f"NI + {label}", cost, t_ms)

    print("\n" + "=" * 68)
    print("PROCESS COMPLETE (All solutions validated)")
    print("=" * 68)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/eil76.tsp...


    Cities (n): 76

[2] Summary Comparison (Optimal: 538):
  Method                             Distance    Time (ms)      Gap %
  -------------------------------------------------------------------
  Nearest Insertion (Base)                620         2.69     15.24%
  NI + 2opt-BI                            579        17.63      7.62%
  NI + 2opt-FI                            565        17.02      5.02%
  NI + Swap-FI                            595       287.67     10.59%
  NI + Swap-BI                            595       464.25     10.59%

PROCESS COMPLETE (All solutions validated)


## **3. Farthest Insertion**

In [38]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE = r"Dataset/eil76.tsp"
    OPTIMAL  = 538

    # [1] Đọc dữ liệu
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # [2] Chạy thuật toán khởi tạo
    t0_fi = time.perf_counter()
    fi_tour, fi_cost = farthest_insertion(instance, start_node=0)
    t_fi = (time.perf_counter() - t0_fi) * 1000

    assert is_valid_tour(fi_tour, n)[0], "Lỗi: Tour khởi tạo không hợp lệ!"

    # Chạy Local Search variants
    ls_configs = [
        ("2opt-BI", two_opt_best_improvement),
        ("2opt-FI", two_opt_first_improvement),
        ("Swap-FI", swap_first_improvement),
        ("Swap-BI", swap_best_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, fi_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [3] Summary Comparison
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<30} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<30} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    _print_row("Farthest Insertion (Base)", fi_cost, t_fi)

    for label, _, cost, t_ms in results:
        _print_row(f"FI + {label}", cost, t_ms)

    print("\n" + "=" * 68)
    print("PROCESS COMPLETE (All solutions validated)")
    print("=" * 68)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/eil76.tsp...
    Cities (n): 76

[2] Summary Comparison (Optimal: 538):
  Method                             Distance    Time (ms)      Gap %
  -------------------------------------------------------------------
  Farthest Insertion (Base)               592         3.53     10.04%
  FI + 2opt-BI                            569        19.57      5.76%
  FI + 2opt-FI                            575         9.75      6.88%
  FI + Swap-FI                            589       107.24      9.48%
  FI + Swap-BI                            589       120.03      9.48%

PROCESS COMPLETE (All solutions validated)


## **4. Cheapest Insertion**

In [39]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE = r"Dataset/eil76.tsp"
    OPTIMAL  = 538

    # [1] Đọc dữ liệu
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # [2] Chạy thuật toán khởi tạo
    t0_ci = time.perf_counter()
    ci_tour, ci_cost = cheapest_insertion(instance, start_node=0)
    t_ci = (time.perf_counter() - t0_ci) * 1000

    assert is_valid_tour(ci_tour, n)[0], "Lỗi: Tour khởi tạo không hợp lệ!"

    # Chạy Local Search variants
    ls_configs = [
        ("2opt-BI", two_opt_best_improvement),
        ("2opt-FI", two_opt_first_improvement),
        ("Swap-FI", swap_first_improvement),
        ("Swap-BI", swap_best_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, ci_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [3] Summary Comparison
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<30} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<30} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    _print_row("Cheapest Insertion (Base)", ci_cost, t_ci)

    for label, _, cost, t_ms in results:
        _print_row(f"CI + {label}", cost, t_ms)

    print("\n" + "=" * 68)
    print("PROCESS COMPLETE (All solutions validated)")
    print("=" * 68)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/eil76.tsp...
    Cities (n): 76

[2] Summary Comparison (Optimal: 538):
  Method                             Distance    Time (ms)      Gap %
  -------------------------------------------------------------------
  Cheapest Insertion (Base)               604        44.62     12.27%
  CI + 2opt-BI                            583        14.68      8.36%
  CI + 2opt-FI                            599        11.09     11.34%
  CI + Swap-FI                            599       167.10     11.34%
  CI + Swap-BI                            600       141.01     11.52%

PROCESS COMPLETE (All solutions validated)


## **5. Greedy Edge**

In [40]:
import time

def main():
    # --- Cấu hình dữ liệu ---
    TSP_FILE = r"Dataset/eil76.tsp"
    OPTIMAL  = 538

    # [1] Đọc dữ liệu
    print(f"\n[1] Reading TSP file: {TSP_FILE}...")
    instance = read_tsp_file(TSP_FILE)
    n = len(instance.nodes)
    print(f"    Cities (n): {n}")

    # [2] Chạy thuật toán khởi tạo: Greedy Edge (Chạy ngầm)
    t0_ge = time.perf_counter()
    ge_tour, ge_cost = greedy_edge(instance)
    t_ge = (time.perf_counter() - t0_ge) * 1000

    assert is_valid_tour(ge_tour, n)[0], "Lỗi: Tour Greedy Edge không hợp lệ!"

    # [3] Chạy Local Search variants (Chạy ngầm)
    ls_configs = [
        ("2opt-FI", two_opt_first_improvement),
        ("2opt-BI", two_opt_best_improvement),
        ("Swap-FI", swap_first_improvement),
        ("Swap-BI", swap_best_improvement),
    ]

    results = []
    for label, func in ls_configs:
        t0_ls = time.perf_counter()
        tour, cost, iters = func(instance, ge_tour[:])
        t_ls = (time.perf_counter() - t0_ls) * 1000

        if not is_valid_tour(tour, n)[0]:
            raise ValueError(f"Lỗi logic tại {label}")

        results.append((label, tour, cost, t_ls))

    # [4] Summary Comparison
    print(f"\n[2] Summary Comparison (Optimal: {OPTIMAL:,}):")
    header = f"  {'Method':<30} {'Distance':>12} {'Time (ms)':>12} {'Gap %':>10}"
    print(header)
    print("  " + "-" * (len(header) - 2))

    def _print_row(name, cost, t_ms):
        gap = (cost - OPTIMAL) / OPTIMAL * 100
        print(f"  {name:<30} {cost:>12,} {t_ms:>12.2f} {gap:>9.2f}%")

    _print_row("Greedy Edge (Base)", ge_cost, t_ge)

    results.sort(key=lambda x: x[2])
    for label, _, cost, t_ms in results:
        _print_row(f"GE + {label}", cost, t_ms)

    best_label, best_tour, best_cost, _ = results[0]
    print(f"\n[3] Best Route Found: GE + {best_label} ({best_cost:,})")

    print("\n" + "=" * 68)
    print("PROCESS COMPLETE (All solutions validated)")
    print("=" * 68)

if __name__ == "__main__":
    main()


[1] Reading TSP file: Dataset/eil76.tsp...
    Cities (n): 76

[2] Summary Comparison (Optimal: 538):
  Method                             Distance    Time (ms)      Gap %
  -------------------------------------------------------------------
  Greedy Edge (Base)                      585         3.61      8.74%
  GE + 2opt-BI                            550        15.68      2.23%
  GE + 2opt-FI                            563        10.13      4.65%
  GE + Swap-FI                            571       482.41      6.13%
  GE + Swap-BI                            571       505.77      6.13%

[3] Best Route Found: GE + 2opt-BI (550)

PROCESS COMPLETE (All solutions validated)
